In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
pip install opencv-python tensorflow mtcnn imageio

In [ ]:
!pip install torch torchvision torchaudio

In [ ]:
!pip install torchmetrics

In [ ]:
pip install decord

In [ ]:
import torch
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from torchmetrics.classification import Accuracy, Precision, F1Score, Recall
import matplotlib.pyplot as plt
import os
from sklearn.utils import resample
from torchvision.io import read_video
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
import numpy as np
import cv2
from tqdm.auto import tqdm


In [ ]:
# model = models.video.mc3_18(pretrained=True)
# model.fc = nn.Sequential(
#     nn.Dropout(p=0.5), 
#     nn.Linear(model.fc.in_features, 2)
# )  

In [ ]:
# for param in model.parameters():
#     param.requires_grad = False

# # Unfreeze the last few layers
# for param in model.layer4.parameters():
#     param.requires_grad = True
# for param in model.layer3.parameters():
#     param.requires_grad = True

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torch.nn.utils import weight_norm

# Define a simple TCN block
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1):
        super().__init__()
        self.conv1 = weight_norm(nn.Conv1d(in_channels, out_channels, kernel_size,
                                           padding=(kernel_size-1) * dilation // 2, dilation=dilation))
        self.relu = nn.ReLU()
        self.conv2 = weight_norm(nn.Conv1d(out_channels, out_channels, kernel_size,
                                           padding=(kernel_size-1) * dilation // 2, dilation=dilation))
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        self.init_weights()

    def init_weights(self):
        nn.init.kaiming_normal_(self.conv1.weight)
        nn.init.kaiming_normal_(self.conv2.weight)
        if self.downsample:
            nn.init.kaiming_normal_(self.downsample.weight)

    def forward(self, x):
        out = self.conv1(x)
        out = self.relu(out)
        out = self.conv2(out)
        if self.downsample:
            x = self.downsample(x)
        return self.relu(out + x)


In [ ]:
# Full TCN model
class TCN(nn.Module):
    def __init__(self, input_size, num_channels, kernel_size=3):
        super().__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            in_channels = input_size if i == 0 else num_channels[i-1]
            layers.append(TemporalBlock(in_channels, num_channels[i], kernel_size, dilation=2**i))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


In [ ]:
class LivenessModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_extractor = models.mobilenet_v2(pretrained=True).features
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.tcn = TCN(input_size=1280, num_channels=[512, 256, 128, 64])
        self.fc = nn.Linear(64, 2)

    def forward(self, x):
        B, T, C, H, W = x.shape
        frame_features = []

        # Process each frame individually
        for t in range(T):
            frame = x[:, t, :, :, :]  # (B, 3, 112, 112)
            # print(f"Input frame shape: {frame.shape}")
            features = self.feature_extractor(frame)  # (B, 1280, 4, 4)
            pooled = self.global_avg_pool(features)  # (B, 1280, 1, 1)
            frame_features.append(pooled.squeeze())  # (B, 1280)

        # Stack and process with TCN
        frame_features = torch.stack(frame_features, dim=1)  # (B, T, 1280)
        x = self.tcn(frame_features.permute(0, 2, 1))  # TCN expects (B, C, T)
        x = torch.mean(x, dim=2)  # (B, 64)
        x = self.fc(x)  # (B, 2)
        return x

In [ ]:
model = LivenessModel()

In [ ]:
device = torch.device("cpu")

In [ ]:

model = model.to(device)


In [ ]:
# import torch
# import torch.nn as nn
# import torchvision.models as models

# class VideoModel(nn.Module):
#     def __init__(self):
#         super(VideoModel, self).__init__()
#         self.feature_extractor = models.video.mc3_18(pretrained=True)  # Replace with a lighter model if needed
#         self.feature_extractor.fc = nn.Identity()  # Remove the final classification layer
        
#         # Temporal Convolutional Network (TCN) for temporal modeling
#         self.tcn = nn.Sequential(
#             nn.Conv1d(in_channels=512, out_channels=256, kernel_size=3, padding=1),
#             nn.ReLU(),
#             nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1),
#             nn.ReLU()
#         )

#         self.fc = nn.Linear(128, 2)  # Final classification layer (2 classes: real/spoof)

    


In [ ]:


# video, _, _= read_video("path", pts_units = "sec")
# video = video.permute(0, 3, 1, 2)
# clip = sample_clip(video)
# clip = torch.stack([transform(frame) for frame in clip])
# clip = clip.permute(1, 0, 2, 3)

In [ ]:
def parsing_rep(base_dir):
    splits = ['train', 'devel', 'test']
    label_map = {'real':0, 'attack': 1}
    dataset = {split: [] for split in splits}

    for split in splits:
        for label_name, label in label_map.items():
            fol_path = os.path.join(base_dir, split, label_name)
            for file in os.listdir(fol_path):
                if file.endswith(".mov"):
                    video_path = os.path.join(fol_path, file)
                    dataset[split].append((video_path, label))
    return dataset

In [ ]:
def parsing_sp(base_dir):
    splits = ['train', 'test']
    label_map = {"real_video": 0, "attack": 1}
    dataset = {split: [] for split in splits}

    for split in splits:
        for label_name, label in label_map.items():
            fol_path = os.path.join(base_dir, split, label_name)
            for file in os.listdir(fol_path):
                if file.endswith(".mp4"):
                    video_path = os.path.join(fol_path, file)
                    dataset[split].append((video_path, label))
    return dataset

In [ ]:
# def extract_frames(video_path, num_frames = 32):
#     cap = cv2.VideoCapture(video_path)
#     total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
#     frame_indices = np.linspace(0, total_frames- 1, num_frames, dtype=int)

#     frames = []
#     for idx in frame_indices:
#         cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
#         ret, frame = cap.read()
#         if ret:
#             frames.append(frames)
#         else:
#             break
#     cap.release()

#     if len(frames) < num_frames:
#         raise ValueError(f"could not extract {num_frames} frames from {video_path}")
#     return np.array(frames)

In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((112, 112)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.5, contrast=0.4, saturation=0.4, hue=0.3),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
def preprocess_frames(frames):
    processed_frames = [transform(frame) for frame in frames]
    return torch.stack(processed_frames).permute(1, 0, 2, 3)    

In [ ]:
def balanced_dataset(train_data):
    class0 = [item for item in train_data if item[1] == 0]
    class1 = [item for item in train_data if item[1] == 1]

    class0_oversampled = resample(class0, replace = True, n_samples = len(class1), random_state = 42)
    balanced_train_data = class0_oversampled + class1
    np.random.shuffle(balanced_train_data)
    return balanced_train_data

In [ ]:
# def to_tens(data, num_frames):
#     videos = []
#     labels = []

#     for video_path , label in data:
#         try:
#             frames = extract_frames(video_path, num_frames = num_frames)
#             video_tensor = preprocess_videos(frames)
#             videos.append(video_tensor.unsqueeze(0))
#             labels.append(label)

#         except Exception as e:
#             print(f"Error Processing {video_path}: {e}")
#             continue


#     labels = torch.tensor(labels)

#     return videos, labels

In [ ]:
class VideoDataset(Dataset):
    def __init__(self, data, num_frames=32, transform=None):
        self.data = data
        self.num_frames = num_frames
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def extract_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_indices = np.linspace(0, total_frames-1, self.num_frames, dtype=int)
        frames = []

        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            # print(f"Frame shape after transform: {frame.shape}")
            if ret:
                frame = cv2.resize(frame, (112, 112))
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                if self.transform:
                    frame = self.transform(frame)  # Should output (3, 112, 112)
                else:
                    frame = torch.from_numpy(frame).permute(2, 0, 1)  # (C, H, W)
                frames.append(frame)
            else:
                break

        cap.release()

        # Pad with zero tensors if needed
        if len(frames) < self.num_frames:
            padding = [torch.zeros(3, 112, 112) for _ in range(self.num_frames - len(frames))]
            frames.extend(padding)

        return torch.stack(frames)  # (T, C, H, W)

    def __getitem__(self, idx):
        video_path, label = self.data[idx]
        frames = self.extract_frames(video_path)
        return frames, torch.tensor(label, dtype=torch.long)

In [ ]:
base_dir_rep = "/kaggle/input/replay/replay-mobile/database"
base_dir_sp = "/kaggle/input/real-vs-fake-anti-spoofing-video-classification"
dataset_rep = parsing_rep(base_dir_rep)
dataset_sp = parsing_sp(base_dir_sp)

all_data = dataset_rep['train'] + dataset_rep['devel'] + dataset_sp['train'] + dataset_sp['test']
train_data, devel_data = train_test_split(all_data, test_size = 0.3, random_state= 42)

train_data = balanced_dataset(train_data)
devel_data = balanced_dataset(devel_data)

train_dataset = VideoDataset(train_data, transform=transform)
val_dataset = VideoDataset(devel_data, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [ ]:
print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))

In [ ]:
# for videos, labels in train_loader:
#     print(videos.shape)  # Should be (4, 32, 3, 112, 112)
#     break

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
loss_fn = torch.nn.CrossEntropyLoss()

accuracy = Accuracy(task = "multiclass", num_classes = 2)
precision = Precision(task = "multiclass", num_classes = 2)
recall = Recall(task = "multiclass", num_classes = 2)
f1 = F1Score(task = "multiclass", num_classes = 2)

train_loss = []
train_acc = []
train_prec = []
train_recall = []
train_f1 = []

val_loss = []
val_acc = []
val_prec = []
val_recall = []
val_f1 = []


In [ ]:
epochs = 2
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    epoch_acc = 0
    epoch_prec = 0
    epoch_rec = 0
    epoch_f1 = 0
    total = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for videos, labels in train_loader:
        
        videos, labels = videos.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(videos)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs, dim = 1)
        acc = accuracy(preds, labels).item()
        prec = precision(preds, labels).item()
        rec = recall(preds, labels).item()
        f1_score = f1(preds, labels).item()
        
        epoch_loss += loss.item()
        epoch_acc += acc
        epoch_prec += prec
        epoch_rec += rec
        epoch_f1 += f1_score
        total += 1
        progress_bar.set_postfix(loss=epoch_loss / total)
        progress_bar.update(1)
        
        del videos, labels, outputs
        torch.cuda.empty_cache()
        
    train_loss.append(epoch_loss/total)
    train_acc.append(epoch_acc/total)
    train_prec.append(epoch_prec/total)
    train_recall.append(epoch_rec/total)
    train_f1.append(epoch_f1/total)

    
    model.eval()
    val_epoch_loss = 0
    val_epoch_acc = 0
    val_epoch_prec = 0
    val_epoch_rec = 0
    val_epoch_f1 = 0
    total = 0

    with torch.no_grad():
        for videos, labels in val_loader:
            videos, labels = videos.to(device), labels.to(device)
            outputs = model(videos)
            loss = loss_fn(outputs, labels)

            preds = torch.argmax(outputs, dim = 1)
            acc = accuracy(preds, labels).item()
            prec = precision(preds, labels).item()
            rec = recall(preds, labels).item()
            f1_score = f1(preds, labels).item()

            val_epoch_loss += loss.item()
            val_epoch_acc += acc
            val_epoch_prec += prec
            val_epoch_rec += rec
            val_epoch_f1 += f1_score
            total += 1

        val_loss.append(val_epoch_loss/ total)
        val_acc.append(val_epoch_acc/ total)
        val_prec.append(val_epoch_prec/total)
        val_recall.append(val_epoch_rec/ total)
        val_f1.append(val_epoch_f1/ total)
    
    print(f"Epoch {epoch+1}/{epochs} - "
        f"Train Loss: {train_loss[-1]:.4f}, Train Acc: {train_acc[-1]:.4f}, Train Prec: {train_prec[-1]:.4f}, "
        f"Train Rec: {train_recall[-1]:.4f}, Train F1: {train_f1[-1]:.4f} | "
        f"Val Loss: {val_loss[-1]:.4f}, Val Acc: {val_acc[-1]:.4f}, Val Prec: {val_prec[-1]:.4f}, "
        f"Val Rec: {val_recall[-1]:.4f}, Val F1: {val_f1[-1]:.4f}")

In [ ]:
import pickle
torch.save(model, 'liveness_final.pth')
pickle.dump(model, open("liveness_final.pkl", 'wb'))

In [ ]:
plt.figure(figsize = (10,5))
plt.plot(train_loss, label = "Loss")
plt.plot(train_acc, label = "accuracy")
plt.plot(train_prec, label = "precision")
plt.xlabel("Epochs")
plt.ylabel('metics')
plt.legend()
plt.show()

In [ ]:
path = '/kaggle/input/'
for root, dirs, files in os.walk(path):
    for name in files:
        if name.endswith(".pth"):
            print(os.path.join(root, name))


In [ ]:
model = torch.load('/kaggle/input/tcn_mobilenet/pytorch/default/1/liveness.pth')

In [ ]:
plt.figure(figsize=(12, 8))

# Plot Recall
plt.subplot(2, 3, 1)
plt.plot(train_recall, label='Train Recall')
plt.plot(val_recall, label='Val Recall')
plt.xlabel('Epochs')
plt.ylabel('Recall')
plt.title('Recall Over Epochs')
plt.legend()

# Plot F1-Score
plt.subplot(2, 3, 2)
plt.plot(train_f1, label='Train F1-Score')
plt.plot(val_f1, label='Val F1-Score')
plt.xlabel('Epochs')
plt.ylabel('F1-Score')
plt.title('F1-Score Over Epochs')
plt.legend()

# Plot Loss
plt.subplot(2, 3, 3)
plt.plot(train_loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss Over Epochs')
plt.legend()

# Plot Accuracy
plt.subplot(2, 3, 4)
plt.plot(train_acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Accuracy Over Epochs')
plt.legend()

# Plot Precision
plt.subplot(2, 3, 5)
plt.plot(train_prec, label='Train Precision')
plt.plot(val_prec, label='Val Precision')
plt.xlabel('Epochs')
plt.ylabel('Precision')
plt.title('Precision Over Epochs')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# all_data = dataset_sp['train'] +  dataset_sp['test']
# train_data, devel_data = train_test_split(all_data, test_size = 0.3, random_state= 42)

# train_data = balanced_dataset(train_data)
# devel_data = balanced_dataset(devel_data)

# train_dataset = VideoDataset(train_data, transform=transform)
# val_dataset = VideoDataset(devel_data, transform=transform)

# train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [ ]:
# print("Train dataset size:", len(train_dataset))
# print("Validation dataset size:", len(val_dataset))

In [ ]:
# optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
# loss_fn = torch.nn.CrossEntropyLoss()

# accuracy = Accuracy(task = "multiclass", num_classes = 2)
# precision = Precision(task = "multiclass", num_classes = 2)
# recall = Recall(task = "multiclass", num_classes = 2)
# f1 = F1Score(task = "multiclass", num_classes = 2)

# train_loss = []
# train_acc = []
# train_prec = []
# train_recall = []
# train_f1 = []

# val_loss = []
# val_acc = []
# val_prec = []
# val_recall = []
# val_f1 = []


In [ ]:
# epochs = 2
# for epoch in range(epochs):
#     model.train()
#     epoch_loss = 0
#     epoch_acc = 0
#     epoch_prec = 0
#     epoch_rec = 0
#     epoch_f1 = 0
#     total = 0
#     progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
#     for videos, labels in train_loader:
        
#         videos, labels = videos.to(device), labels.to(device)
#         optimizer.zero_grad()
#         outputs = model(videos)
#         loss = loss_fn(outputs, labels)
#         loss.backward()
#         optimizer.step()

#         preds = torch.argmax(outputs, dim = 1)
#         acc = accuracy(preds, labels).item()
#         prec = precision(preds, labels).item()
#         rec = recall(preds, labels).item()
#         f1_score = f1(preds, labels).item()
        
#         epoch_loss += loss.item()
#         epoch_acc += acc
#         epoch_prec += prec
#         epoch_rec += rec
#         epoch_f1 += f1_score
#         total += 1
#         progress_bar.set_postfix(loss=epoch_loss / total)
#         progress_bar.update(1)
        
#         del videos, labels, outputs
#         torch.cuda.empty_cache()
        
#     train_loss.append(epoch_loss/total)
#     train_acc.append(epoch_acc/total)
#     train_prec.append(epoch_prec/total)
#     train_recall.append(epoch_rec/total)
#     train_f1.append(epoch_f1/total)

    
#     model.eval()
#     val_epoch_loss = 0
#     val_epoch_acc = 0
#     val_epoch_prec = 0
#     val_epoch_rec = 0
#     val_epoch_f1 = 0
#     total = 0

#     with torch.no_grad():
#         for videos, labels in val_loader:
#             videos, labels = videos.to(device), labels.to(device)
#             outputs = model(videos)
#             loss = loss_fn(outputs, labels)

#             preds = torch.argmax(outputs, dim = 1)
#             acc = accuracy(preds, labels).item()
#             prec = precision(preds, labels).item()
#             rec = recall(preds, labels).item()
#             f1_score = f1(preds, labels).item()

#             val_epoch_loss += loss.item()
#             val_epoch_acc += acc
#             val_epoch_prec += prec
#             val_epoch_rec += rec
#             val_epoch_f1 += f1_score
#             total += 1

#         val_loss.append(val_epoch_loss/ total)
#         val_acc.append(val_epoch_acc/ total)
#         val_prec.append(val_epoch_prec/total)
#         val_recall.append(val_epoch_rec/ total)
#         val_f1.append(val_epoch_f1/ total)
    
#     print(f"Epoch {epoch+1}/{epochs} - "
#         f"Train Loss: {train_loss[-1]:.4f}, Train Acc: {train_acc[-1]:.4f}, Train Prec: {train_prec[-1]:.4f}, "
#         f"Train Rec: {train_recall[-1]:.4f}, Train F1: {train_f1[-1]:.4f} | "
#         f"Val Loss: {val_loss[-1]:.4f}, Val Acc: {val_acc[-1]:.4f}, Val Prec: {val_prec[-1]:.4f}, "
#         f"Val Rec: {val_recall[-1]:.4f}, Val F1: {val_f1[-1]:.4f}")

In [ ]:
import pickle
torch.save(model, 'livenesshp.pth')
pickle.dump(model, open("livenesshp.pkl", 'wb'))

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)
# crit = torch.nn.CrossEntropyLoss()

In [ ]:
# import cv2
# import numpy as np
# import torch
# from torch.utils.data import Dataset

# class VideoDataset(Dataset):
#     def __init__(self, data, num_frames=32, transform=None, is_test=False):
#         self.data = data  # For test: list of video paths. For train: list of (path, label)
#         self.num_frames = num_frames
#         self.transform = transform
#         self.is_test = is_test

#     def __len__(self):
#         return len(self.data)

#     def extract_frames(self, video_path):
#         if not isinstance(video_path, str):
#             raise TypeError(f"video_path must be a string. Got: {type(video_path)}")

#         # Verify the file exists
#         if not os.path.exists(video_path):
#             raise FileNotFoundError(f"Video not found: {video_path}")

#         cap = cv2.VideoCapture(video_path)
#         total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
#         frame_indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)
#         frames = []

#         for idx in frame_indices:
#             cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
#             ret, frame = cap.read()
#             if ret:
#                 frame = cv2.resize(frame, (112, 112))
#                 frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#                 frame = torch.tensor(frame, dtype=torch.float32).permute(2, 0, 1)  # [C, H, W]
#                 frame /= 255.0  # Normalize to [0, 1]
#                 if self.transform:
#                     frame = self.transform(frame)
#                 frames.append(frame)
#             else:
#                 break

#         cap.release()

#         # Pad with zeros if needed
#         if len(frames) < self.num_frames:
#             pad_frame = torch.zeros(3, 112, 112, dtype=torch.float32)
#             frames.extend([pad_frame] * (self.num_frames - len(frames)))

#         # Shape: [T, C, H, W] (Time, Channels, Height, Width)
#         return torch.stack(frames)

#     def __getitem__(self, idx):
#         if self.is_test:
#             # Extract video path even if data is accidentally a tuple
#             item = self.data[idx]
#             if isinstance(item, tuple):
#                 video_path = item[0]  # Assume path is the first element
#             else:
#                 video_path = item
#             frames = self.extract_frames(video_path)
#             return frames, video_path
#         else:
#             # Training/validation: explicit (path, label) tuples
#             video_path, label = self.data[idx]
#             frames = self.extract_frames(video_path)
#             return frames, torch.tensor(label, dtype=torch.long)


In [ ]:
# def test_collate(batch):
#     # Separate frames and video paths
#     frames = torch.stack([item[0] for item in batch])  # Shape: [B, T, C, H, W]
#     paths = [item[1] for item in batch]  # List of video paths
#     return frames, paths

In [ ]:
# test_dataset = VideoDataset(dataset_rep['test'] + dataset_sp["test"], is_test = True)
# test_loader = DataLoader(test_dataset, batch_size = 16, shuffle = False, collate_fn = test_collate)


In [ ]:
# accuracy = Accuracy(task = "multiclass", num_classes = 2)
# precision = Precision(task = "multiclass", num_classes = 2)
# recall = Recall(task = "multiclass", num_classes = 2)
# f1 = F1Score(task = "multiclass", num_classes = 2)
# running_loss = 0.0
# with torch.no_grad():
#     for videos, labels in tqdm(test_loader, desc="Testing"):
#         outputs = model(videos)
#         loss = crit(outputs, labels)
#         running_loss += loss.item()

#         # Update all metrics
#         accuracy.update(outputs, labels)
#         precision.update(outputs, labels)
#         recall.update(outputs, labels)
#         f1.update(outputs, labels)

# # Compute final results
# test_acc = accuracy.compute() * 100
# test_precision = precision.compute() * 100
# test_recall = recall.compute() * 100
# test_f1 = f1.compute() * 100


# print(f"Accuracy: {test_acc:.2f}%")
# print(f"Precision: {test_precision:.2f}%")
# print(f"Recall: {test_recall:.2f}%")
# print(f"F1 Score: {test_f1:.2f}%")
